# SPEACH TO TEXT

In [2]:
%pip install faster-whisper

  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 5.7 MB/s  0:00:00
   ---------------------------------------- 0.0/18.8 MB ? eta -:--:--
   ---- ----------------------------------- 2.1/18.8 MB 9.7 MB/s eta 0:00:02
   --------- ------------------------------ 4.5/18.8 MB 9.9 MB/s eta 0:00:02
   ------------- -------------------------- 6.6/18.8 MB 10.3 MB/s eta 0:00:02
   ---------------- ----------------------- 7.9/18.8 MB 9.1 MB/s eta 0:00:02
   -------------------- ------------------- 9.4/18.8 MB 8.6 MB/s eta 0:00:02
   ---------------------- ----------------- 10.7/18.8 MB 8.4 MB/s eta 0:00:01
   -------------------------- ------------- 12.3/18.8 MB 8.1 MB/s eta 0:00:01
   ----------------------------- ---------- 13.9/18.8 MB 8.0 MB/s eta 0:00:01
   -----------------------------

In [13]:
import pandas as pd
import os
from faster_whisper import WhisperModel

def transcribe_and_append(audio_path, csv_path, target_column, model=None):
    """
    Transcribes an audio file and appends the result to a specific column in a CSV.
    
    Parameters:
    - audio_path (str): Path to the audio file (e.g., .mp3, .wav).
    - csv_path (str): Path to the CSV file.
    - target_column (str): The name of the column where the text should be stored.
    - model (WhisperModel, optional): A pre-loaded faster-whisper model object.
    """
    
    # 1. Load the model if one wasn't passed in
    # (Using large-v3-turbo and float16 optimized for your hardware)
    if model is None:
        print("Loading Whisper model into VRAM...")
        model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")

    # 2. Transcribe the audio
    print(f"Transcribing '{os.path.basename(audio_path)}'...")
    segments, info = model.transcribe(audio_path, beam_size=5)

    # Combine the generator segments into a single continuous string
    transcription = " ".join([segment.text for segment in segments]).strip()
    print(f"Transcription complete. Length: {len(transcription)} characters.")

    # 3. Handle the CSV updating
    file_exists = os.path.isfile(csv_path)
    
    if file_exists:
        df = pd.read_csv(csv_path)
    else:
        # If the file doesn't exist, initialize an empty DataFrame
        df = pd.DataFrame()

    # Create a new row of data. 
    # Adding the filename as an ID is best practice so you know which audio it came from.
    new_row_data = {
        # "Audio_File": os.path.basename(audio_path), 
        target_column: transcription
    }
    
    # Convert the new row into a DataFrame and concatenate it
    new_row_df = pd.DataFrame([new_row_data])
    df = pd.concat([df, new_row_df], ignore_index=True)

    # Save the updated DataFrame back to the CSV
    df.to_csv(csv_path, index=False)
    print(f"Successfully appended data to '{csv_path}'.\n")
    
    return transcription

In [16]:
df1 = pd.read_csv('clipped.csv')
df1.columns

Index(['Unnamed: 0', 'Date received', 'Product', 'Sub-product', 'Issue',
       'Sub-issue', 'Consumer complaint narrative', 'Company public response',
       'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='str')

In [4]:
import os
# This disables the symlink behavior entirely
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

# 2. Transcribe the audio
model = WhisperModel("large-v3-turbo", device="cpu", compute_type="int8") # using cpu here
audio_path = 'sample.m4a'
print(f"Transcribing '{os.path.basename(audio_path)}'...")
segments, info = model.transcribe(audio_path, beam_size=5)

# Combine the generator segments into a single continuous string
transcription = " ".join([segment.text for segment in segments]).strip()
print(f"Transcription complete. Length: {len(transcription)} characters.")

print(transcription)

Transcribing 'sample.m4a'...
Transcription complete. Length: 45 characters.
Can you please close my bank account? Please.


In [21]:
# ==========================================
# Example Usage:
# ==========================================

# Load the model ONCE outside the function to save time on multiple files
my_model = WhisperModel("large-v3-turbo", device="cpu", compute_type="int8") # using cpu here

# Run the function
transcribe_and_append(
    audio_path="sample.m4a",
    csv_path="clipped.csv",
    target_column="Consumer complaint narrative",
    model=my_model
)


Transcribing 'sample.m4a'...
Transcription complete. Length: 45 characters.
Successfully appended data to 'clipped.csv'.



'Can you please close my bank account? Please.'

In [26]:
# removing appended
df = pd.read_csv('clipped.csv')

In [19]:
df = df[:-1]

In [27]:

print(df.iloc[-1])

Unnamed: 0                                                                NaN
Date received                                                             NaN
Product                                                                   NaN
Sub-product                                                               NaN
Issue                                                                     NaN
Sub-issue                                                                 NaN
Consumer complaint narrative    Can you please close my bank account? Please.
Company public response                                                   NaN
Company                                                                   NaN
State                                                                     NaN
ZIP code                                                                  NaN
Tags                                                                      NaN
Consumer consent provided?                                      